# WTA Match Predictor - Model Training

## Contents

- Import Data and Packages
- Preprocess Data and Define Features
- Train, Test, and Compare Models

### Import Data and Required Packages

In [153]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns # for scrambling columns
import random
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

### Import CSV Data as DataFrame

In [154]:
df = pd.read_csv('data/stud.csv')
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2023-609,Indian Wells,Hard,96,PM,20230306,286,214544,2.0,NaN,...,58.0,37.0,16.0,13.0,8.0,11.0,2,6100,16,2205
1,2023-609,Indian Wells,Hard,96,PM,20230306,285,216347,1.0,NaN,...,55.0,30.0,4.0,10.0,3.0,8.0,1,10585,36,1303
2,2023-609,Indian Wells,Hard,96,PM,20230306,284,221054,NaN,NaN,...,52.0,37.0,10.0,12.0,5.0,8.0,77,784,13,2246
3,2023-609,Indian Wells,Hard,96,PM,20230306,283,201514,NaN,NaN,...,24.0,14.0,12.0,8.0,0.0,5.0,83,743,43,1200
4,2023-609,Indian Wells,Hard,96,PM,20230306,282,201614,5.0,NaN,...,50.0,34.0,21.0,14.0,1.0,4.0,5,4905,49,1080


### Preprocessing

In [155]:
# First, we want to drop columns that we know will not be useful for our model or understanding.
# Information related to the tournament is not relevant
df.drop(columns=['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date'], axis=1, inplace=True)
# Certain information about players (e.g. height, age, hand) are also irrelevant
df.drop(columns=['winner_hand', 'loser_hand', 'winner_ht', 'loser_ht', 'winner_age', 'loser_age', 'winner_ioc', 'loser_ioc'], axis=1, inplace=True)
# Certain match information will also be irrelevant for predictions
df.drop(columns=['match_num', 'best_of', 'round', 'minutes', 'score'], axis=1, inplace=True)

In [156]:
df.head()

,winner_id,winner_seed,winner_entry,winner_name,loser_id,loser_seed,loser_entry,loser_name,w_ace,w_df,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,214544,2.0,NaN,Aryna Sabalenka,206252,16.0,NaN,Barbora Krejcikova,11.0,4.0,...,58.0,37.0,16.0,13.0,8.0,11.0,2,6100,16,2205
1,216347,1.0,NaN,Iga Swiatek,215370,32.0,NaN,Bianca Andreescu,1.0,0.0,...,55.0,30.0,4.0,10.0,3.0,8.0,1,10585,36,1303
2,221054,NaN,NaN,Emma Raducanu,206242,13.0,NaN,Beatriz Haddad Maia,2.0,4.0,...,52.0,37.0,10.0,12.0,5.0,8.0,77,784,13,2246
3,201514,NaN,NaN,Sorana Cirstea,211095,NaN,NaN,Bernarda Pera,4.0,0.0,...,24.0,14.0,12.0,8.0,0.0,5.0,83,743,43,1200
4,201614,5.0,NaN,Caroline Garcia,220367,30.0,NaN,Leylah Fernandez,11.0,3.0,...,50.0,34.0,21.0,14.0,1.0,4.0,5,4905,49,1080


Something very important to note is that, if we implicitly define matches as already having a winner and loser, our model will always predict the match outcome with 100% accuracy, since the data will always indicate who wins and who loses. To fix this, we need to scramble the columns for winners and losers, so it's not obvious to the model who wins the match. We will also rename columns so that there is no mention of a winner or a loser, and will instead encode winner/loser as a single number, 0 (for player 1) or 1 (for player 2).

In [ ]:
# First: Rename 'winner_'/'w_' and 'loser_'/'l_' columns to represent player 1 and player 2 respectively
for col in df.columns:
    if 'winner' in col:
        df.rename(columns={col: col.replace('winner', 'p1')}, inplace=True)
    elif 'w' in col:
        df.rename(columns={col: col.replace('w', 'p1')}, inplace=True)
    elif 'loser' in col:
        df.rename(columns={col: col.replace('loser', 'p2')}, inplace=True)
    elif 'l' in col:
        df.rename(columns={col: col.replace('l', 'p2')}, inplace=True)
df['match_winner'] = 0 # create match winner column with all 0s (player 1 for all obs.)

In [158]:
df.head()

,p1_id,p1_seed,p1_entry,p1_name,p2_id,p2_seed,p2_entry,p2_name,p1_ace,p1_df,...,p2_1stWon,p2_2ndWon,p2_SvGms,p2_bpSaved,p2_bpFaced,p1_rank,p1_rank_points,p2_rank,p2_rank_points,match_winner
0,214544,2.0,NaN,Aryna Sabalenka,206252,16.0,NaN,Barbora Krejcikova,11.0,4.0,...,37.0,16.0,13.0,8.0,11.0,2,6100,16,2205,0
1,216347,1.0,NaN,Iga Swiatek,215370,32.0,NaN,Bianca Andreescu,1.0,0.0,...,30.0,4.0,10.0,3.0,8.0,1,10585,36,1303,0
2,221054,NaN,NaN,Emma Raducanu,206242,13.0,NaN,Beatriz Haddad Maia,2.0,4.0,...,37.0,10.0,12.0,5.0,8.0,77,784,13,2246,0
3,201514,NaN,NaN,Sorana Cirstea,211095,NaN,NaN,Bernarda Pera,4.0,0.0,...,14.0,12.0,8.0,0.0,5.0,83,743,43,1200,0
4,201614,5.0,NaN,Caroline Garcia,220367,30.0,NaN,Leylah Fernandez,11.0,3.0,...,34.0,21.0,14.0,1.0,4.0,5,4905,49,1080,0


In [159]:
# Second: Randomly shuffle p1 and p2 columns to ensure no pattern
cols1 = [c for c in df.columns if 'p1' in c]
cols2 = [c for c in df.columns if 'p2' in c]
cols2_targ = [c.replace('p1', 'p2') for c in cols1]
print(cols1)
print(cols2)

# copy df for scrambling
copy = df.copy()

#maskIdx = [i for i in df.index if i % 3]
# generate len(df.index) / 2 random indices. for those observations, swap columns and set match_winner to 1
maskIdx = [random.randint(0, len(df.index) - 1) for i in range((int)(len(df.index) / 2))]
print(maskIdx)
df.loc[maskIdx, cols1] = copy.loc[maskIdx, cols2_targ].values
df.loc[maskIdx, cols2_targ] = copy.loc[maskIdx, cols1].values
df.loc[maskIdx, 'match_winner'] = 1

['p1_id', 'p1_seed', 'p1_entry', 'p1_name', 'p1_ace', 'p1_df', 'p1_svpt', 'p1_1stIn', 'p1_1stWon', 'p1_2ndWon', 'p1_SvGms', 'p1_bpSaved', 'p1_bpFaced', 'p1_rank', 'p1_rank_points']
['p2_id', 'p2_seed', 'p2_entry', 'p2_name', 'p2_ace', 'p2_df', 'p2_svpt', 'p2_1stIn', 'p2_1stWon', 'p2_2ndWon', 'p2_SvGms', 'p2_bpSaved', 'p2_bpFaced', 'p2_rank', 'p2_rank_points']
[15, 88, 35, 4, 26, 31, 9, 54, 52, 79, 86, 23, 0, 18, 0, 108, 112, 31, 29, 68, 29, 53, 95, 68, 10, 24, 80, 16, 54, 5, 43, 46, 72, 46, 1, 43, 29, 53, 9, 80, 50, 30, 97, 14, 48, 50, 107, 18, 57, 1, 39, 28, 7, 3, 89, 73]


In [160]:
df.head(50)

,p1_id,p1_seed,p1_entry,p1_name,p2_id,p2_seed,p2_entry,p2_name,p1_ace,p1_df,...,p2_1stWon,p2_2ndWon,p2_SvGms,p2_bpSaved,p2_bpFaced,p1_rank,p1_rank_points,p2_rank,p2_rank_points,match_winner
0,206252,16.0,NaN,Barbora Krejcikova,214544,2.0,NaN,Aryna Sabalenka,3.0,6.0,...,39.0,12.0,14.0,1.0,4.0,16,2205,2,6100,1
1,215370,32.0,NaN,Bianca Andreescu,216347,1.0,NaN,Iga Swiatek,1.0,2.0,...,31.0,10.0,11.0,4.0,8.0,36,1303,1,10585,1
2,221054,NaN,NaN,Emma Raducanu,206242,13.0,NaN,Beatriz Haddad Maia,2.0,4.0,...,37.0,10.0,12.0,5.0,8.0,77,784,13,2246,0
3,211095,NaN,NaN,Bernarda Pera,201514,NaN,NaN,Sorana Cirstea,3.0,5.0,...,19.0,12.0,8.0,0.0,1.0,43,1200,83,743,1
4,220367,30.0,NaN,Leylah Fernandez,201614,5.0,NaN,Caroline Garcia,5.0,8.0,...,42.0,26.0,15.0,2.0,2.0,49,1080,5,4905,1
5,202460,4.0,NaN,Ons Jabeur,214954,NaN,NaN,Marketa Vondrousova,2.0,2.0,...,23.0,17.0,11.0,0.0,4.0,4,4921,105,632,1
6,214096,NaN,NaN,Karolina Muchova,203354,23.0,NaN,Martina Trevisan,7.0,7.0,...,41.0,17.0,14.0,10.0,13.0,76,791,26,1529,0
7,211651,21.0,NaN,Paula Badosa,214981,10.0,NaN,Elena Rybakina,2.0,7.0,...,30.0,19.0,10.0,5.0,6.0,22,1758,10,2935,1
8,201514,NaN,NaN,Sorana Cirstea,214096,NaN,Q,Karolina Muchova,4.0,1.0,...,20.0,10.0,9.0,3.0,8.0,74,838,55,1006,0
9,201662,17.0,NaN,Karolina Pliskova,214954,NaN,NaN,Marketa Vondrousova,2.0,5.0,...,23.0,6.0,8.0,4.0,6.0,17,2155,103,632,1
